# Token Decompression

## Learning Objectives
1. Identify frequent token spans (bigrams/trigrams) as compression candidates
2. Implement a TokenCompressor that maps spans to mega-token IDs
3. Measure compression ratio and roundtrip accuracy
4. Compare different decompression strategies and their trade-offs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
from collections import Counter
from itertools import islice

## Level 1: Bigram Frequency Analysis

In [ ]:
# Simulate a code corpus and identify high-frequency token spans
corpus_lines = [
    "def train model optimizer dataloader",
    "for batch in dataloader optimizer zero_grad",
    "loss = model batch loss backward",
    "optimizer step scheduler step",
    "def evaluate model test_loader",
    "model eval total_loss = 0",
    "with torch no_grad output = model x",
    "return total_loss / len test_loader",
    "for epoch in range num_epochs",
    "train_loss = train model optimizer dataloader",
] * 20   # repeat to build frequency

all_tokens = []
for line in corpus_lines:
    all_tokens.extend(line.split())

def get_ngrams(tokens, n):
    counts = Counter()
    for i in range(len(tokens) - n + 1):
        counts[tuple(tokens[i:i+n])] += 1
    return counts

unigrams = Counter(all_tokens)
bigrams  = get_ngrams(all_tokens, 2)
trigrams = get_ngrams(all_tokens, 3)

total = len(all_tokens)
print(f"Total tokens: {total}")
print(f"Unique unigrams: {len(unigrams)}")
print(f"Unique bigrams:  {len(bigrams)}")

# Compression value: count × (span_length - 1) tokens saved per compression
def compression_value(span, count):
    return count * (len(span) - 1)

all_candidates = []
for span, count in list(bigrams.items()) + list(trigrams.items()):
    if count >= 5:
        all_candidates.append((span, count, compression_value(span, count)))

all_candidates.sort(key=lambda x: x[2], reverse=True)

print(f"\nTop 10 compression candidates:")
print(f"{'Span':<45} {'Count':<8} {'Tokens saved'}")
print("-" * 65)
for span, count, saved in all_candidates[:10]:
    print(f"{' '.join(span):<45} {count:<8} {saved}")

# Simulate applying top-5 compressions
top5 = [c[0] for c in all_candidates[:5]]
total_savings = sum(c[2] for c in all_candidates[:5])
print(f"\nOriginal tokens:             {total}")
print(f"Savings from top-5 spans:    {total_savings}")
print(f"Compressed total:            {total - total_savings}  ({(total-total_savings)/total:.1%})")

## Level 2: TokenCompressor with Roundtrip

In [ ]:
class TokenCompressor:
    """
    Map frequent bigrams/trigrams to single mega-token IDs.
    Supports encode (compress) and decode (decompress) for roundtrip.
    """
    def __init__(self, vocab_size=500, min_count=5, max_span=3):
        self.vocab_size    = vocab_size
        self.min_count     = min_count
        self.max_span      = max_span
        self.span_to_id    = {}   # span_tuple -> mega_token_id
        self.id_to_span    = {}   # mega_token_id -> span_tuple
        self.next_id       = vocab_size

    def fit(self, sequences: List[List[int]]):
        """Learn frequent spans from integer-tokenised sequences."""
        counts = Counter()
        for seq in sequences:
            for n in range(2, self.max_span + 1):
                for i in range(len(seq) - n + 1):
                    counts[tuple(seq[i:i+n])] += 1
        for span, count in counts.most_common():
            if count < self.min_count:
                break
            self.span_to_id[span] = self.next_id
            self.id_to_span[self.next_id] = span
            self.next_id += 1
        return self

    def encode(self, seq: List[int]) -> List[int]:
        """Greedily compress longest matching spans."""
        compressed = []
        i = 0
        while i < len(seq):
            matched = False
            for n in range(self.max_span, 1, -1):
                if i + n <= len(seq):
                    span = tuple(seq[i:i+n])
                    if span in self.span_to_id:
                        compressed.append(self.span_to_id[span])
                        i += n
                        matched = True
                        break
            if not matched:
                compressed.append(seq[i])
                i += 1
        return compressed

    def decode(self, compressed: List[int]) -> List[int]:
        """Expand mega-tokens back to original token IDs."""
        original = []
        for tok in compressed:
            if tok in self.id_to_span:
                original.extend(self.id_to_span[tok])
            else:
                original.append(tok)
        return original

    def compression_stats(self, sequences: List[List[int]]):
        orig_total  = sum(len(s) for s in sequences)
        comp_total  = sum(len(self.encode(s)) for s in sequences)
        ratio       = 1 - comp_total / orig_total
        return {'original': orig_total, 'compressed': comp_total, 'ratio': ratio}

# Generate synthetic sequences with known patterns
rng = np.random.default_rng(42)
PATTERNS  = [(10, 20), (30, 40), (50, 60), (10, 20, 30), (70, 80)]
sequences = []
for _ in range(200):
    seq = []
    while len(seq) < 60:
        if rng.random() < 0.45:
            p = PATTERNS[int(rng.integers(len(PATTERNS)))]
            seq.extend(p)
        else:
            seq.append(int(rng.integers(100)))
    sequences.append(seq[:60])

comp = TokenCompressor(vocab_size=100, min_count=8, max_span=3)
comp.fit(sequences)
stats = comp.compression_stats(sequences)

print(f"Learned {len(comp.span_to_id)} mega-tokens")
print(f"Original tokens:   {stats['original']}")
print(f"Compressed tokens: {stats['compressed']}")
print(f"Compression ratio: {stats['ratio']:.1%}")

# Roundtrip verification
n_errors = 0
for seq in sequences:
    encoded = comp.encode(seq)
    decoded = comp.decode(encoded)
    if decoded != seq:
        n_errors += 1

print(f"\nRoundtrip errors: {n_errors}/{len(sequences)}  (should be 0)")

# Show a sample
sample = sequences[0][:20]
enc    = comp.encode(sample)
dec    = comp.decode(enc)
print(f"\nSample (first 20 tokens):")
print(f"  Original  ({len(sample):2d}): {sample}")
print(f"  Encoded   ({len(enc):2d}): {enc}")
print(f"  Decoded   ({len(dec):2d}): {dec}")
print(f"  Match: {dec == sample}")
# ─── Extended compression analysis ───────────────────────────────────
print("\n=== Compression Ratio vs Pattern Frequency ===")
np.random.seed(0)
for pattern_prob9 in [0.1, 0.2, 0.3, 0.4, 0.5]:
    PATS9 = [(10, 20), (30, 40), (50, 60)]
    seqs9 = []
    rng9  = np.random.default_rng(int(pattern_prob9*100))
    for _ in range(100):
        seq9 = []
        while len(seq9) < 50:
            if rng9.random() < pattern_prob9:
                seq9.extend(PATS9[int(rng9.integers(len(PATS9)))])
            else:
                seq9.append(int(rng9.integers(80)))
        seqs9.append(seq9[:50])
    comp9 = TokenCompressor(vocab_size=80, min_count=5, max_span=2)
    comp9.fit(seqs9)
    stats9 = comp9.compression_stats(seqs9)
    # Verify roundtrip
    err9 = sum(1 for s in seqs9 if comp9.decode(comp9.encode(s)) != s)
    print(f"  pattern_prob={pattern_prob9:.1f}: ratio={stats9['ratio']:.2%}, "
          f"mega_tokens={len(comp9.span_to_id)}, errors={err9}")

print("\n=== BPE Merge Vocabulary Growth ===")
rng9b = np.random.default_rng(42)
corpus9 = []
for _ in range(50):
    seq9b = []
    while len(seq9b) < 30:
        if rng9b.random() < 0.4: seq9b.extend([10,20])
        else: seq9b.append(int(rng9b.integers(25)))
    corpus9.append(seq9b[:30])

print(f"{'n_merges':<12} {'Vocab size':<14} {'Avg seq len':<14} {'Comp ratio'}")
print("-" * 52)
for n_m9 in [0, 5, 10, 15, 20]:
    bpe9 = BPESimulator(vocab_size=25)
    comp_seqs9 = bpe9.fit(corpus9, n_merges=n_m9)
    orig9  = sum(len(s) for s in corpus9)
    comp9b = sum(len(s) for s in comp_seqs9)
    vocab9 = 25 + n_m9
    print(f"  {n_m9:<12} {vocab9:<14} {comp9b/len(corpus9):<14.1f} {1-comp9b/orig9:.2%}")

print("\n=== Speculative Decoding: Draft k Sweep ===")
print(f"{'k draft':<10} {'Acceptance':<14} {'Speedup est':<14} {'Verify calls/100'}")
print("-" * 52)
for k9, rate9 in [(1,0.9),(2,0.85),(4,0.75),(6,0.7),(8,0.65)]:
    s9 = SpeculativeDecoder(k_draft=k9, accept_rate=rate9)
    r9 = s9.decode(n_tokens=100, rng=np.random.default_rng(42))
    print(f"  {k9:<10} {r9['acceptance_rate']:<14.3f} {r9['speedup_estimate']:<14.2f} {r9['verify_calls']}")


## Real-World Example 1: Speculative Decoding Token Reuse

In [ ]:
# Speculative decoding: draft model generates k tokens, verifier accepts/rejects
# Token decompression: accepted spans are 'decompressed' into full tokens

class SpeculativeDecoder:
    """
    Simulate speculative decoding with a draft model (fast, small) and
    a verifier model (slow, large).
    Draft model proposes k tokens; verifier accepts a prefix of them.
    """
    def __init__(self, vocab_size=100, k_draft=4, accept_rate=0.75):
        self.vocab     = vocab_size
        self.k         = k_draft
        self.accept_rate = accept_rate

    def draft(self, context: List[int], rng) -> List[int]:
        """Draft k tokens autoregressively (simulated)."""
        return [int(rng.integers(self.vocab)) for _ in range(self.k)]

    def verify(self, context: List[int], draft_tokens: List[int], rng) -> int:
        """
        Returns how many draft tokens the verifier accepts.
        In practice: sample from verifier; if it matches draft, accept.
        """
        n_accepted = 0
        for tok in draft_tokens:
            if rng.random() < self.accept_rate:
                n_accepted += 1
            else:
                break
        return n_accepted

    def decode(self, n_tokens=50, rng=None):
        rng = rng or np.random.default_rng(42)
        context = [0]   # BOS
        total_draft_calls  = 0
        total_verify_calls = 0
        total_accepted     = 0

        while len(context) - 1 < n_tokens:
            draft = self.draft(context, rng)
            total_draft_calls += self.k
            n_acc  = self.verify(context, draft, rng)
            total_verify_calls += 1
            # Accept the n_acc prefix; verifier generates 1 correction token
            context.extend(draft[:n_acc])
            total_accepted += n_acc
            # Verifier generates 1 corrected token regardless
            context.append(int(rng.integers(self.vocab)))
            total_draft_calls += 0  # verifier's 1 token is 'free' in this model

        return {
            'sequence_length':  len(context) - 1,
            'verify_calls':     total_verify_calls,
            'acceptance_rate':  total_accepted / max(total_draft_calls, 1),
            'speedup_estimate': (total_accepted + total_verify_calls) / max(total_verify_calls, 1)
        }

spec = SpeculativeDecoder(k_draft=4, accept_rate=0.75)
rng  = np.random.default_rng(0)
result = spec.decode(n_tokens=100, rng=rng)

print("Speculative decoding statistics:")
for k, v in result.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

# Sweep acceptance rate
print("\nAcceptance rate sweep (k=4 draft tokens):")
print(f"{'Accept rate':<14} {'Speedup':<12} {'Verify calls/100tok'}")
print("-" * 42)
for rate in [0.5, 0.65, 0.75, 0.85, 0.95]:
    s = SpeculativeDecoder(k_draft=4, accept_rate=rate)
    r = s.decode(n_tokens=100, rng=np.random.default_rng(1))
    print(f"{rate:<14.2f} {r['speedup_estimate']:<12.2f} {r['verify_calls']}")

## Real-World Example 2: BPE Merge Simulation

In [ ]:
# BPE (Byte Pair Encoding): iteratively merge the most frequent pair
# Token decompression = the inverse of BPE merging

class BPESimulator:
    def __init__(self, vocab_size=30):
        self.merges   = []   # list of (pair, merged_token)
        self.vocab    = list(range(vocab_size))   # base vocab
        self.next_tok = vocab_size

    def fit(self, corpus: List[List[int]], n_merges=20):
        seqs = [list(s) for s in corpus]
        for step in range(n_merges):
            counts = Counter()
            for seq in seqs:
                for i in range(len(seq)-1):
                    counts[(seq[i], seq[i+1])] += 1
            if not counts:
                break
            best_pair = counts.most_common(1)[0][0]
            new_tok   = self.next_tok
            self.merges.append((best_pair, new_tok))
            self.vocab.append(new_tok)
            self.next_tok += 1
            # Apply merge
            new_seqs = []
            for seq in seqs:
                new_seq = []
                i = 0
                while i < len(seq):
                    if i + 1 < len(seq) and (seq[i], seq[i+1]) == best_pair:
                        new_seq.append(new_tok)
                        i += 2
                    else:
                        new_seq.append(seq[i])
                        i += 1
                new_seqs.append(new_seq)
            seqs = new_seqs
        return seqs   # compressed sequences

    def decompress(self, seq: List[int]) -> List[int]:
        """Reverse all BPE merges to recover original byte-level tokens."""
        merge_map = {new: pair for pair, new in self.merges}

        def expand(tok):
            if tok in merge_map:
                a, b = merge_map[tok]
                return expand(a) + expand(b)
            return [tok]

        result = []
        for tok in seq:
            result.extend(expand(tok))
        return result

rng = np.random.default_rng(42)
# Simulate byte-level tokens (0-29) with frequent patterns
base_corpus = []
for _ in range(100):
    seq = []
    while len(seq) < 40:
        if rng.random() < 0.4:
            seq.extend([10, 20])           # frequent pair
        elif rng.random() < 0.3:
            seq.extend([5, 15, 25])        # frequent triple
        else:
            seq.append(int(rng.integers(30)))
    base_corpus.append(seq[:40])

bpe = BPESimulator(vocab_size=30)
compressed = bpe.fit(base_corpus, n_merges=15)

orig_len = sum(len(s) for s in base_corpus)
comp_len = sum(len(s) for s in compressed)
print(f"BPE training: {len(bpe.merges)} merges learned")
print(f"Corpus tokens: {orig_len} -> {comp_len}  ({comp_len/orig_len:.1%})")

# Verify roundtrip
errors = 0
for orig, comp in zip(base_corpus, compressed):
    decompressed = bpe.decompress(comp)
    if decompressed != orig:
        errors += 1
print(f"Roundtrip errors: {errors}/{len(base_corpus)}")

# Show merge sequence
print("\nTop 5 learned merges:")
for pair, new_tok in bpe.merges[:5]:
    print(f"  {pair[0]} + {pair[1]} -> {new_tok}")

# ─── Extended token compression analysis ─────────────────────────────
print("\n=== Compression Efficiency: Span Length vs Frequency ===")
rng44 = np.random.default_rng(0)
PATTERNS44 = [(10,20), (30,40), (50,60), (10,20,30), (70,80,90,100)]
seqs44 = []
for _ in range(300):
    seq44 = []
    while len(seq44) < 80:
        if rng44.random() < 0.4:
            p44 = PATTERNS44[int(rng44.integers(len(PATTERNS44)))]
            seq44.extend(p44)
        else:
            seq44.append(int(rng44.integers(120)))
    seqs44.append(seq44[:80])

comp44 = TokenCompressor(vocab_size=120, min_count=10, max_span=4)
comp44.fit(seqs44)

print(f"{'Span length':<14} {'# mega-tokens':<16} {'Avg compression/span'}")
print("-" * 48)
by_len44 = {}
for span44, mid44 in comp44.span_to_id.items():
    l44 = len(span44)
    by_len44.setdefault(l44, []).append(span44)
for l44, spans44 in sorted(by_len44.items()):
    savings44 = l44 - 1  # tokens saved per use
    print(f"  {l44:<14} {len(spans44):<16} {savings44} tokens/use")

# Total compression on seqs44
stats44 = comp44.compression_stats(seqs44)
print(f"\nTotal compression: {stats44['ratio']:.2%}")
err44 = sum(1 for s in seqs44 if comp44.decode(comp44.encode(s)) != s)
print(f"Roundtrip errors: {err44}/{len(seqs44)}")

print("\n=== BPE: Subword Vocabulary Distribution ===")
rng44b = np.random.default_rng(1)
corpus44 = []
for _ in range(200):
    seq44b = []
    while len(seq44b) < 40:
        if rng44b.random() < 0.45: seq44b.extend([10,20])
        elif rng44b.random() < 0.3: seq44b.extend([5,15,25])
        else: seq44b.append(int(rng44b.integers(30)))
    corpus44.append(seq44b[:40])

bpe44 = BPESimulator(vocab_size=30)
comp_corpus44 = bpe44.fit(corpus44, n_merges=20)

orig44 = sum(len(s) for s in corpus44)
comp44b = sum(len(s) for s in comp_corpus44)
print(f"  Merges: {len(bpe44.merges)}, Vocab: {len(bpe44.vocab)}")
print(f"  Corpus: {orig44} -> {comp44b} tokens ({comp44b/orig44:.1%})")

# Merge frequency distribution
merge_saves44 = Counter()
for orig_seq44, comp_seq44 in zip(corpus44, comp_corpus44):
    reduction44 = len(orig_seq44) - len(comp_seq44)
    for pair44, _ in bpe44.merges:
        # Crude estimate: how many times was this merge applied?
        for i in range(len(orig_seq44)-1):
            if (orig_seq44[i], orig_seq44[i+1]) == pair44:
                merge_saves44[pair44] += 1
print(f"  Top merge by application count: {merge_saves44.most_common(3)}")


## Real-World Example 3: Context Compression (LLMLingua-Style)

In [ ]:
# LLMLingua: compress prompt tokens by removing unimportant ones
# Measure perplexity increase as quality metric

class LLMLinguaCompressor:
    """
    Token-level prompt compression: score each token's importance via
    a small language model, remove low-scoring tokens.
    """
    def __init__(self, vocab_size=100, embed_dim=32, hidden_dim=64):
        self.model = nn.Sequential(
            nn.Embedding(vocab_size, embed_dim),
            nn.LSTM(embed_dim, hidden_dim, batch_first=True),
        ).to(device)
        # Importance scorer: small MLP on hidden states
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        ).to(device)
        self.vocab = vocab_size

    def score_tokens(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Returns importance score [0,1] per token."""
        with torch.no_grad():
            embs, (h, _) = self.model[0](token_ids), None
            # Simple: use embedding norm as proxy for importance
            embs  = self.model[0](token_ids)        # [B, T, E]
            scores = embs.norm(dim=-1, keepdim=True)  # [B, T, 1]
            scores = self.scorer(scores.squeeze(-1).unsqueeze(-1))   # needs [B, T, hidden]
        return scores.squeeze(-1)   # [B, T]

    def compress(self, token_ids: torch.Tensor, keep_ratio: float = 0.5) -> torch.Tensor:
        """Keep only the top keep_ratio tokens by importance score."""
        B, T = token_ids.shape
        # Use embedding magnitude as importance proxy (no trained scorer needed)
        embs  = self.model[0](token_ids)    # [B, T, E]
        imps  = embs.norm(dim=-1)           # [B, T]
        k     = max(1, int(T * keep_ratio))
        top_k_idx = imps.topk(k, dim=-1).indices  # [B, k]
        top_k_idx, _ = top_k_idx.sort(dim=-1)     # preserve order
        compressed = torch.stack([token_ids[b, top_k_idx[b]] for b in range(B)])
        return compressed, top_k_idx

compressor = LLMLinguaCompressor()
B, T = 4, 64
tokens = torch.randint(0, 100, (B, T), device=device)

print("LLMLingua-style compression:")
for ratio in [1.0, 0.75, 0.50, 0.25]:
    if ratio == 1.0:
        comp, idx = tokens, torch.arange(T, device=device).unsqueeze(0).expand(B,-1)
    else:
        comp, idx = compressor.compress(tokens, keep_ratio=ratio)
    print(f"  keep={ratio:.0%}: {tokens.shape} -> {comp.shape}")

# Measure how well important tokens preserve information
orig_embs = compressor.model[0](tokens).norm(dim=-1)  # [B, T]
comp50, idx50 = compressor.compress(tokens, 0.5)
kept_mass = orig_embs.gather(1, idx50).sum() / orig_embs.sum()
print(f"\nImportance mass retained at 50% compression: {kept_mass.item():.3f}")

## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies  = ['No compress','BPE\n(standard)','Bigram\nmega-tok','Speculative\ndecoding','LLMLingua\n50%']
comp_ratio  = [0, 30, 20, 45, 50]   # % token reduction
quality_loss = [0, 0, 0, 1, 5]     # % quality degradation
colors      = ['#2196F3','#4CAF50','#43A047','#9C27B0','#FF9800']

bars = ax1.bar(strategies, comp_ratio, color=colors)
ax1.set_ylabel('Token reduction (%)')
ax1.set_title('Compression Ratio by Strategy')
ax1.set_ylim(0, 65)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, comp_ratio):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v}%', ha='center', fontsize=9)

ax2.scatter(comp_ratio, quality_loss, c=colors, s=180, zorder=5)
for i, s in enumerate(strategies):
    ax2.annotate(s.replace('\n',' '), (comp_ratio[i], quality_loss[i]),
                 textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Token reduction (%)'); ax2.set_ylabel('Quality degradation (%)')
ax2.set_title('Quality vs Compression')
ax2.axhline(3, color='red', linestyle='--', alpha=0.5, label='3% quality floor')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/44-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Token decompression is the inverse of token compression — expanding compact representations back to their original token sequences, enabling efficient storage or transmission while preserving lossless or near-lossless reconstruction.

### Variants and When to Use

| Method | Use When | Trade-off | Compression |
|--------|----------|-----------|------------|
| BPE merges | Standard tokenisation (always) | None — lossless | 20-40% |
| Bigram mega-tokens | Domain-specific corpora | Vocab inflation | 15-25% |
| Speculative decoding | Autoregressive inference speed | Needs separate draft model | 2-4x speedup |
| LLMLingua | Long prompt compression | Lossy, affects recall | 50-80% |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Roundtrip errors | Merge order not respected in decode | Apply merges in reverse order |
| Low speculative acceptance | Draft model too different from verifier | Use same family, smaller variant |
| LLMLingua drops critical tokens | Scorer not calibrated to task | Use task-specific calibration set |

## Exercises

1. **Roundtrip test:** Add a 4-gram pattern to `PATTERNS` in Level 2 and verify the compressor still achieves 100% roundtrip accuracy.
2. **BPE sweep:** Run the BPE simulator with n_merges ∈ {5, 10, 20, 50} and plot compression ratio vs roundtrip errors.
3. **Speculative acceptance:** Modify `SpeculativeDecoder` to use k=8 draft tokens instead of k=4 and compare the speedup estimate.